In [40]:
# This notebook constructs MEPI indicators, calculates MEPI scores, and derives target variables using weighted assignments
import pandas as pd
import numpy as np

df = pd.read_parquet("PAKISTAN/DATA/1.df_PK_NecessaryColumns.parquet", engine="pyarrow")

# 1) Basic shape
print("Rows, Cols:", df.shape)

# 2) Household ID checks
print("\nID missing:", df["hhidCaseIdentification"].isna().sum())
print("ID duplicates:", df["hhidCaseIdentification"].duplicated().sum())

# Show duplicate examples (if any)
dups = df[df["hhidCaseIdentification"].duplicated(keep=False)].sort_values("hhidCaseIdentification")
print("\nDuplicate ID sample (first 10 rows):")
print(dups[["hhidCaseIdentification","hv000CountryCode","hv001ClusterNumber","hv002HouseholdNumber"]].head(10))


Rows, Cols: (14540, 28)

ID missing: 0
ID duplicates: 0

Duplicate ID sample (first 10 rows):
Empty DataFrame
Columns: [hhidCaseIdentification, hv000CountryCode, hv001ClusterNumber, hv002HouseholdNumber]
Index: []


In [42]:
print(df["hv270WealthIndex"].value_counts(dropna=False).sort_index())
print("dtype:", df["hv270WealthIndex"].dtype)
#1    2608  → Poorest
#2    2627  → Poorer
#3    2525  → Middle
#4    2458  → Richer
#5    2282  → Richest
#dtype: int8

hv270WealthIndex
1    2837
2    3104
3    2815
4    2720
5    3064
Name: count, dtype: int64
dtype: int8


In [44]:
print(df["hv206HasElectricity"].value_counts(dropna=False))
print("dtype:", df["hv206HasElectricity"].dtype)

hv206HasElectricity
1.0    13541
0.0      995
NaN        4
Name: count, dtype: int64
dtype: float64


In [48]:
print(df["hv226CookingFuel"].value_counts(dropna=False))
print("dtype:", df["hv226CookingFuel"].dtype)
#{"1": "electricity", "2": "lpg", "3": "natural gas", "4": "biogas", "5": "kerosene",
 #"6": "coal, lignite", "7": "charcoal", "8": "wood", "9": "straw/shrubs/grass", 
 #"10": "agricultural crop", "11": "animal dung", "95": "no food cooked in house", "96": "other"}

hv226CookingFuel
8.0     6538
3.0     5102
2.0     1904
11.0     345
7.0      238
9.0      121
95.0      84
10.0      83
4.0       75
1.0       30
6.0       14
96.0       2
NaN        2
5.0        2
Name: count, dtype: int64
dtype: float64


In [50]:
print(df["hv242SeparateKitchenYesNo"].value_counts(dropna=False))
print("dtype:", df["hv242SeparateKitchenYesNo"].dtype)

hv242SeparateKitchenYesNo
1.0    9706
0.0    3558
NaN    1276
Name: count, dtype: int64
dtype: float64


In [52]:
print(df["hv241FoodCookedInHouse"].value_counts(dropna=False))
print("dtype:", df["hv241FoodCookedInHouse"].dtype)

hv241FoodCookedInHouse
1.0    13268
2.0     1125
NaN       86
3.0       60
6.0        1
Name: count, dtype: int64
dtype: float64


In [54]:
print(df["hv209HasRefrigerator"].value_counts(dropna=False))
print("dtype:", df["hv209HasRefrigerator"].dtype)

hv209HasRefrigerator
1.0    8228
0.0    6307
NaN       5
Name: count, dtype: int64
dtype: float64


In [56]:
print(df["hv208HasTelevision"].value_counts(dropna=False))
print("dtype:", df["hv208HasTelevision"].dtype)

hv208HasTelevision
1.0    9091
0.0    5445
NaN       4
Name: count, dtype: int64
dtype: float64


In [58]:
print(df["hv243aHasMobilePhone"].value_counts(dropna=False))
print("dtype:", df["hv243aHasMobilePhone"].dtype)

hv243aHasMobilePhone
1.0    13666
0.0      869
NaN        5
Name: count, dtype: int64
dtype: float64


In [60]:
print(df["hv221HasLandLine"].value_counts(dropna=False))
print("dtype:", df["hv221HasLandLine"].dtype)

hv221HasLandLine
0.0    13259
1.0     1277
NaN        4
Name: count, dtype: int64
dtype: float64


In [62]:
# 2) Create deprivation indicators (1=deprived, 0=not deprived)
# -------------------------
# Electricity (1=yes, 0=no)
df["Depr_Elec"] = np.where(df["hv206HasElectricity"] == 1, 0,
                    np.where(df["hv206HasElectricity"] == 0, 1, np.nan))

In [64]:
list(df.columns)

['hhidCaseIdentification',
 'hv000CountryCode',
 'hv001ClusterNumber',
 'hv002HouseholdNumber',
 'hv005SimplilingWeight',
 'hv270WealthIndex',
 'hv206HasElectricity',
 'hv226CookingFuel',
 'hv241FoodCookedInHouse',
 'hv242SeparateKitchenYesNo',
 'hv209HasRefrigerator',
 'hv221HasLandLine',
 'hv243aHasMobilePhone',
 'hv208HasTelevision',
 'hv024RegionDivision',
 'hv025TypeOfResidence',
 'hv219SexOfHead',
 'hv220AgeOfHead',
 'hv106_01EducationOfHead',
 'hv115_01MaritalStatus',
 'hv009FamilySize',
 'hv247HasBankAccount',
 'hv216HouseSize',
 'hv014NoOfChildren',
 'v714CurrentlyWorking',
 'v717Occupation',
 '745aHouseOwnership',
 'weight',
 'Depr_Elec']

In [66]:
print(df["hv206HasElectricity"].value_counts(dropna=False))
print("dtype:", df["hv206HasElectricity"].dtype)

hv206HasElectricity
1.0    13541
0.0      995
NaN        4
Name: count, dtype: int64
dtype: float64


In [68]:
print(df["Depr_Elec"].value_counts(dropna=False))
print("dtype:", df["Depr_Elec"].dtype)

Depr_Elec
0.0    13541
1.0      995
NaN        4
Name: count, dtype: int64
dtype: float64


In [70]:
# Cooking fuel: NOT deprived if using electricity, gas (LPG+natural gas), kerosene, or biogas
CLEAN_FUEL_CODES = {1, 2, 3, 4, 5}

# Cooking fuel (clean -> 0, other -> 1)
df["Depr_CleanFuel"] = np.where(df["hv226CookingFuel"].isin(CLEAN_FUEL_CODES), 0,
                         np.where(df["hv226CookingFuel"].isna(), np.nan, 1))

In [72]:
#print(df["hv226CookingFuel"].value_counts(dropna=False))
#print("dtype:", df["hv226CookingFuel"].dtype)
CLEAN_FUEL_CODES = {1, 2, 3, 4, 5}
print (df["hv226CookingFuel"].isin(CLEAN_FUEL_CODES).sum())
print (df["hv226CookingFuel"].isin(CLEAN_FUEL_CODES).eq(False).sum())
#{"1": "electricity", "2": "lpg", "3": "natural gas", "4": "biogas", "5": "kerosene",
 #"6": "coal, lignite", "7": "charcoal", "8": "wood", "9": "straw/shrubs/grass", 
 #"10": "agricultural crop", "11": "animal dung", "95": "no food cooked in house", "96": "other"}

print(df["Depr_CleanFuel"].value_counts(dropna=False))
print("dtype:", df["Depr_CleanFuel"].dtype)

7113
7427
Depr_CleanFuel
1.0    7425
0.0    7113
NaN       2
Name: count, dtype: int64
dtype: float64


In [74]:
# Refrigerator
df["Depr_Refrigerator"] = np.where(df["hv209HasRefrigerator"] == 1, 0,
                            np.where(df["hv209HasRefrigerator"] == 0, 1, np.nan))

In [76]:
print(df["hv209HasRefrigerator"].value_counts(dropna=False))
print("dtype:", df["hv209HasRefrigerator"].dtype)

print(df["Depr_Refrigerator"].value_counts(dropna=False))
print("dtype:", df["Depr_Refrigerator"].dtype)

hv209HasRefrigerator
1.0    8228
0.0    6307
NaN       5
Name: count, dtype: int64
dtype: float64
Depr_Refrigerator
0.0    8228
1.0    6307
NaN       5
Name: count, dtype: int64
dtype: float64


In [78]:
# Television
df["Depr_TV"] = np.where(df["hv208HasTelevision"] == 1, 0,
                  np.where(df["hv208HasTelevision"] == 0, 1, np.nan))

In [82]:
print(df["hv208HasTelevision"].value_counts(dropna=False))
print("dtype:", df["hv208HasTelevision"].dtype)

print(df["Depr_TV"].value_counts(dropna=False))
print("dtype:", df["Depr_TV"].dtype)

hv208HasTelevision
1.0    9091
0.0    5445
NaN       4
Name: count, dtype: int64
dtype: float64
Depr_TV
0.0    9091
1.0    5445
NaN       4
Name: count, dtype: int64
dtype: float64


In [84]:
# Phone (deprived only if NO mobile AND NO landline)
df["Depr_Phone"] = np.where(
    df["hv243aHasMobilePhone"].isna() | df["hv221HasLandLine"].isna(),
    np.nan,
    np.where((df["hv243aHasMobilePhone"] == 0) & (df["hv221HasLandLine"] == 0), 1, 0)
)

In [86]:
list(df.columns)

['hhidCaseIdentification',
 'hv000CountryCode',
 'hv001ClusterNumber',
 'hv002HouseholdNumber',
 'hv005SimplilingWeight',
 'hv270WealthIndex',
 'hv206HasElectricity',
 'hv226CookingFuel',
 'hv241FoodCookedInHouse',
 'hv242SeparateKitchenYesNo',
 'hv209HasRefrigerator',
 'hv221HasLandLine',
 'hv243aHasMobilePhone',
 'hv208HasTelevision',
 'hv024RegionDivision',
 'hv025TypeOfResidence',
 'hv219SexOfHead',
 'hv220AgeOfHead',
 'hv106_01EducationOfHead',
 'hv115_01MaritalStatus',
 'hv009FamilySize',
 'hv247HasBankAccount',
 'hv216HouseSize',
 'hv014NoOfChildren',
 'v714CurrentlyWorking',
 'v717Occupation',
 '745aHouseOwnership',
 'weight',
 'Depr_Elec',
 'Depr_CleanFuel',
 'Depr_Refrigerator',
 'Depr_TV',
 'Depr_Phone']

In [88]:
print(df["Depr_Phone"].value_counts(dropna=False))
print("dtype:", df["Depr_Phone"].dtype)

Depr_Phone
0.0    13727
1.0      807
NaN        6
Name: count, dtype: int64
dtype: float64


In [90]:
# Kitchen (hv242 preferred; fallback hv241FoodCookedInHouse)
df["Depr_Kitchen"] = np.nan

# hv242SeparateKitchenYesNo: 1 yes (not deprived), 0 no (deprived)
df.loc[df["hv242SeparateKitchenYesNo"] == 1, "Depr_Kitchen"] = 0
df.loc[df["hv242SeparateKitchenYesNo"] == 0, "Depr_Kitchen"] = 1

# fallback to hv241FoodCookedInHouse when hv242 missing
mask = df["hv242SeparateKitchenYesNo"].isna()

In [92]:
print(df["Depr_Kitchen"].value_counts(dropna=False))
print("dtype:", df["Depr_Kitchen"].dtype)

#print(mask)

Depr_Kitchen
0.0    9706
1.0    3558
NaN    1276
Name: count, dtype: int64
dtype: float64


In [94]:
# hv241FoodCookedInHouse: 2 separate building -> not deprived
df.loc[mask & (df["hv241FoodCookedInHouse"] == 2), "Depr_Kitchen"] = 0
# 1 in the house, 3 outdoors, 6 other -> deprived
df.loc[mask & df["hv241FoodCookedInHouse"].isin([1, 3, 6]), "Depr_Kitchen"] = 1

In [96]:
print(df["Depr_Kitchen"].value_counts(dropna=False))
print("dtype:", df["Depr_Kitchen"].dtype)

Depr_Kitchen
0.0    10831
1.0     3623
NaN       86
Name: count, dtype: int64
dtype: float64


In [98]:
list(df.columns)

['hhidCaseIdentification',
 'hv000CountryCode',
 'hv001ClusterNumber',
 'hv002HouseholdNumber',
 'hv005SimplilingWeight',
 'hv270WealthIndex',
 'hv206HasElectricity',
 'hv226CookingFuel',
 'hv241FoodCookedInHouse',
 'hv242SeparateKitchenYesNo',
 'hv209HasRefrigerator',
 'hv221HasLandLine',
 'hv243aHasMobilePhone',
 'hv208HasTelevision',
 'hv024RegionDivision',
 'hv025TypeOfResidence',
 'hv219SexOfHead',
 'hv220AgeOfHead',
 'hv106_01EducationOfHead',
 'hv115_01MaritalStatus',
 'hv009FamilySize',
 'hv247HasBankAccount',
 'hv216HouseSize',
 'hv014NoOfChildren',
 'v714CurrentlyWorking',
 'v717Occupation',
 '745aHouseOwnership',
 'weight',
 'Depr_Elec',
 'Depr_CleanFuel',
 'Depr_Refrigerator',
 'Depr_TV',
 'Depr_Phone',
 'Depr_Kitchen']

In [110]:
cols = [
    'Depr_Elec',
    'Depr_CleanFuel',
    'Depr_Kitchen',
    'Depr_Refrigerator',
    'Depr_TV',
    'Depr_Phone'
]

df[cols].isna().sum()

Depr_Elec             4
Depr_CleanFuel        2
Depr_Kitchen         86
Depr_Refrigerator     5
Depr_TV               4
Depr_Phone            6
dtype: int64

In [112]:
df = df.dropna(subset=cols)

In [114]:
weights = {
    "Depr_Elec": 0.20,
    "Depr_CleanFuel": 0.20,
    "Depr_Kitchen": 0.15,
    "Depr_Refrigerator": 0.15,
    "Depr_TV": 0.15,
    "Depr_Phone": 0.15
  }

In [116]:
mepi_cols = list(weights.keys())
print(mepi_cols)

['Depr_Elec', 'Depr_CleanFuel', 'Depr_Kitchen', 'Depr_Refrigerator', 'Depr_TV', 'Depr_Phone']


In [118]:
# Convert to numeric 0/1 (in case they are 'yes'/'no' or categories)
for col in mepi_cols:
    df[col] = df[col].astype(int)

# Quick check
for col in mepi_cols:
    print(col)
    print(df[col].value_counts(dropna=False), "\n")


Depr_Elec
Depr_Elec
0    13457
1      989
Name: count, dtype: int64 

Depr_CleanFuel
Depr_CleanFuel
1    7338
0    7108
Name: count, dtype: int64 

Depr_Kitchen
Depr_Kitchen
0    10824
1     3622
Name: count, dtype: int64 

Depr_Refrigerator
Depr_Refrigerator
0    8206
1    6240
Name: count, dtype: int64 

Depr_TV
Depr_TV
0    9051
1    5395
Name: count, dtype: int64 

Depr_Phone
Depr_Phone
0    13647
1      799
Name: count, dtype: int64 



In [120]:
df["MEPI_score"] = np.nan
df['MEPI_score'] = sum(df[col] * w for col, w in weights.items())

In [122]:
print(df["MEPI_score"].value_counts(dropna=False))

MEPI_score
0.00    4614
0.50    2062
0.35    1564
0.15    1560
0.20    1426
0.65    1240
0.30     678
0.85     415
0.70     397
0.45     180
0.80     169
1.00      93
0.55      25
0.60      20
0.40       3
Name: count, dtype: int64


In [124]:
# get dependent varialble Energy Poor
ROUND_DECIMALS = 4  # Adjust based on the precision needs
CUTOFF = 0.35

df['EnergyPoor'] = (df['MEPI_score'].round(ROUND_DECIMALS) >= CUTOFF).astype(float)
df['EnergyPoor'] = df['EnergyPoor'].where(df['MEPI_score'].notna(), np.nan)

# Verification
print(f"Energy Poor: {(df['EnergyPoor'] == 1).sum():,}")
print(f"Not Energy Poor: {(df['EnergyPoor'] == 0).sum():,}")
print(f"Missing: {df['EnergyPoor'].isna().sum():,}")

# Should output:
# Energy Poor: 6,168
# Not Energy Poor: 8,278
# Missing: [whatever missing MEPI scores you have]

Energy Poor: 6,168
Not Energy Poor: 8,278
Missing: 0


In [126]:
list(df.columns)

['hhidCaseIdentification',
 'hv000CountryCode',
 'hv001ClusterNumber',
 'hv002HouseholdNumber',
 'hv005SimplilingWeight',
 'hv270WealthIndex',
 'hv206HasElectricity',
 'hv226CookingFuel',
 'hv241FoodCookedInHouse',
 'hv242SeparateKitchenYesNo',
 'hv209HasRefrigerator',
 'hv221HasLandLine',
 'hv243aHasMobilePhone',
 'hv208HasTelevision',
 'hv024RegionDivision',
 'hv025TypeOfResidence',
 'hv219SexOfHead',
 'hv220AgeOfHead',
 'hv106_01EducationOfHead',
 'hv115_01MaritalStatus',
 'hv009FamilySize',
 'hv247HasBankAccount',
 'hv216HouseSize',
 'hv014NoOfChildren',
 'v714CurrentlyWorking',
 'v717Occupation',
 '745aHouseOwnership',
 'weight',
 'Depr_Elec',
 'Depr_CleanFuel',
 'Depr_Refrigerator',
 'Depr_TV',
 'Depr_Phone',
 'Depr_Kitchen',
 'MEPI_score',
 'EnergyPoor']

In [128]:
df.to_parquet(
    "PAKISTAN/DATA/2.df_PK_MEPI_Score_Energy_Poor.parquet",engine="pyarrow",compression="snappy")
print(f"✅ File saved.")

✅ File saved.
